In [1]:
import os
import numpy as np
import pandas as pd

BASE_PATH = "/kaggle/data/champs-scalar-coupling"
TRAIN_PATH = os.path.join(BASE_PATH, "train.csv")
TEST_PATH = os.path.join(BASE_PATH, "test.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
STRUCTURES_PATH = os.path.join(BASE_PATH, "structures.csv")

assert os.path.exists(TRAIN_PATH), f"Missing: {TRAIN_PATH}"
assert os.path.exists(TEST_PATH), f"Missing: {TEST_PATH}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing: {SAMPLE_SUB_PATH}"
assert os.path.exists(STRUCTURES_PATH), f"Missing: {STRUCTURES_PATH}"

print("Found files:")
print(" -", TRAIN_PATH)
print(" -", TEST_PATH)
print(" -", SAMPLE_SUB_PATH)
print(" -", STRUCTURES_PATH)



Found files:
 - /kaggle/data/champs-scalar-coupling/train.csv
 - /kaggle/data/champs-scalar-coupling/test.csv
 - /kaggle/data/champs-scalar-coupling/sample_submission.csv
 - /kaggle/data/champs-scalar-coupling/structures.csv


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

required_train_cols = {
    "id",
    "molecule_name",
    "atom_index_0",
    "atom_index_1",
    "type",
    "scalar_coupling_constant",
}
required_test_cols = {"id", "molecule_name", "atom_index_0", "atom_index_1", "type"}
required_sub_cols = {"id", "scalar_coupling_constant"}

missing_train = required_train_cols - set(train.columns)
missing_test = required_test_cols - set(test.columns)
missing_sub = required_sub_cols - set(sample_sub.columns)

assert not missing_train, f"train.csv missing columns: {missing_train}"
assert not missing_test, f"test.csv missing columns: {missing_test}"
assert not missing_sub, f"sample_submission.csv missing columns: {missing_sub}"

for df in (train, test):
    df["molecule_name"] = df["molecule_name"].astype("string")
    df["atom_index_0"] = pd.to_numeric(df["atom_index_0"], downcast="integer")
    df["atom_index_1"] = pd.to_numeric(df["atom_index_1"], downcast="integer")
    df["type"] = df["type"].astype("string")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_sub.shape)
print("train types:", train["type"].nunique(), "unique")



train shape: (4191263, 6)
test shape: (467813, 5)
sample_submission shape: (467813, 2)
train types: 8 unique


In [3]:
structures = pd.read_csv(
    STRUCTURES_PATH, usecols=["molecule_name", "atom_index", "atom", "x", "y", "z"]
)

structures["molecule_name"] = structures["molecule_name"].astype("string")
structures["atom_index"] = pd.to_numeric(structures["atom_index"], downcast="integer")

structures["x"] = structures["x"].astype(np.float32, copy=False)
structures["y"] = structures["y"].astype(np.float32, copy=False)
structures["z"] = structures["z"].astype(np.float32, copy=False)
structures["atom"] = structures["atom"].astype("string")

structures = structures.sort_values(
    ["molecule_name", "atom_index"], kind="mergesort"
).reset_index(drop=True)


def add_atom_and_distance_features(df, structures_df):
    """
    Core logic preserved (merge structures, compute distances, create bins/keys).
    """
    df = df.copy()
    df["molecule_name"] = df["molecule_name"].astype("string")
    df["atom_index_0"] = pd.to_numeric(df["atom_index_0"], downcast="integer")
    df["atom_index_1"] = pd.to_numeric(df["atom_index_1"], downcast="integer")

    # Keep original direction for directed distance features
    df["atom_index_0_orig"] = df["atom_index_0"]
    df["atom_index_1_orig"] = df["atom_index_1"]

    # Canonicalize indices for undirected distance / unordered pair features
    a0 = df["atom_index_0"].astype("int64")
    a1 = df["atom_index_1"].astype("int64")
    df["atom_index_0"] = np.minimum(a0, a1).astype(df["atom_index_0"].dtype, copy=False)
    df["atom_index_1"] = np.maximum(a0, a1).astype(df["atom_index_1"].dtype, copy=False)

    s0 = structures_df.rename(
        columns={
            "atom_index": "atom_index_0",
            "atom": "atom_0_canon",
            "x": "x0",
            "y": "y0",
            "z": "z0",
        }
    )
    df = df.merge(s0, on=["molecule_name", "atom_index_0"], how="left", sort=False)

    s1 = structures_df.rename(
        columns={
            "atom_index": "atom_index_1",
            "atom": "atom_1_canon",
            "x": "x1",
            "y": "y1",
            "z": "z1",
        }
    )
    df = df.merge(s1, on=["molecule_name", "atom_index_1"], how="left", sort=False)

    dx = df["x0"].astype("float64") - df["x1"].astype("float64")
    dy = df["y0"].astype("float64") - df["y1"].astype("float64")
    dz = df["z0"].astype("float64") - df["z1"].astype("float64")
    df["dist"] = np.sqrt(dx * dx + dy * dy + dz * dz)

    a0s = df["atom_0_canon"].astype("string")
    a1s = df["atom_1_canon"].astype("string")
    df["atom_pair_0"] = a0s.where(a0s <= a1s, a1s)
    df["atom_pair_1"] = a1s.where(a0s <= a1s, a0s)

    # Directed (original order) atom labels
    s0o = structures_df.rename(
        columns={
            "atom_index": "atom_index_0_orig",
            "atom": "atom_0",
        }
    )[["molecule_name", "atom_index_0_orig", "atom_0"]]
    df = df.merge(
        s0o, on=["molecule_name", "atom_index_0_orig"], how="left", sort=False
    )

    s1o = structures_df.rename(
        columns={
            "atom_index": "atom_index_1_orig",
            "atom": "atom_1",
        }
    )[["molecule_name", "atom_index_1_orig", "atom_1"]]
    df = df.merge(
        s1o, on=["molecule_name", "atom_index_1_orig"], how="left", sort=False
    )

    # Directed (original order) coordinates for directed distance
    sd0 = structures_df.rename(
        columns={
            "atom_index": "atom_index_0_orig",
            "x": "x0o",
            "y": "y0o",
            "z": "z0o",
        }
    )[["molecule_name", "atom_index_0_orig", "x0o", "y0o", "z0o"]]
    df = df.merge(
        sd0, on=["molecule_name", "atom_index_0_orig"], how="left", sort=False
    )

    sd1 = structures_df.rename(
        columns={
            "atom_index": "atom_index_1_orig",
            "x": "x1o",
            "y": "y1o",
            "z": "z1o",
        }
    )[["molecule_name", "atom_index_1_orig", "x1o", "y1o", "z1o"]]
    df = df.merge(
        sd1, on=["molecule_name", "atom_index_1_orig"], how="left", sort=False
    )

    dxo = df["x0o"].astype("float64") - df["x1o"].astype("float64")
    dyo = df["y0o"].astype("float64") - df["y1o"].astype("float64")
    dzo = df["z0o"].astype("float64") - df["z1o"].astype("float64")
    df["dist_dir"] = np.sqrt(dxo * dxo + dyo * dyo + dzo * dzo)

    return df


train_f = add_atom_and_distance_features(train, structures)
test_f = add_atom_and_distance_features(test, structures)

global_bin_width = 0.05
type_dist_std = (
    train_f.groupby("type", observed=True)["dist"]
    .std()
    .replace([0.0, np.inf, -np.inf], np.nan)
)
global_std = float(train_f["dist"].std())
if not np.isfinite(global_std) or global_std <= 0:
    global_std = 1.0

type_bin_width = (global_bin_width * (type_dist_std / global_std)).clip(
    lower=0.02, upper=0.20
)

bw_train = (
    train_f["type"].map(type_bin_width).astype("float64").fillna(global_bin_width)
)
bw_test = test_f["type"].map(type_bin_width).astype("float64").fillna(global_bin_width)

train_f["dist_bin"] = np.floor(train_f["dist"] / bw_train).astype("Int32")
test_f["dist_bin"] = np.floor(test_f["dist"] / bw_test).astype("Int32")

SENTINEL_BIN = -1
train_f["dist_bin"] = train_f["dist_bin"].fillna(SENTINEL_BIN).astype("Int32")
test_f["dist_bin"] = test_f["dist_bin"].fillna(SENTINEL_BIN).astype("Int32")

SENTINEL_QBIN = -1

N_QBINS = 64
N_QBINS_COARSE = 16

# Quantile edges for UNDIRECTED distance
edges_by_type = {}
edges_by_type_coarse = {}
for t, grp in train_f.groupby("type", observed=True):
    d = grp["dist"].to_numpy(dtype="float64", copy=False)
    d = d[np.isfinite(d)]
    if d.size < 50:
        edges_by_type[t] = None
        edges_by_type_coarse[t] = None
        continue

    qs_f = np.linspace(0.0, 1.0, N_QBINS + 1)
    edges_f = np.quantile(d, qs_f)
    edges_f = np.unique(edges_f)
    edges_by_type[t] = edges_f if edges_f.size > 2 else None

    qs_c = np.linspace(0.0, 1.0, N_QBINS_COARSE + 1)
    edges_c = np.quantile(d, qs_c)
    edges_c = np.unique(edges_c)
    edges_by_type_coarse[t] = edges_c if edges_c.size > 2 else None

# Change (score-relevant, minimal): build SEPARATE quantile edges for DIRECTED distance.
# Previously, dist_dir bins used edges learned from dist, which mis-bins directed distances
# and reduces match-rate/quality for the top-of-chain directed lookups.
edges_by_type_dir = {}
edges_by_type_dir_coarse = {}
for t, grp in train_f.groupby("type", observed=True):
    d = grp["dist_dir"].to_numpy(dtype="float64", copy=False)
    d = d[np.isfinite(d)]
    if d.size < 50:
        edges_by_type_dir[t] = None
        edges_by_type_dir_coarse[t] = None
        continue

    qs_f = np.linspace(0.0, 1.0, N_QBINS + 1)
    edges_f = np.quantile(d, qs_f)
    edges_f = np.unique(edges_f)
    edges_by_type_dir[t] = edges_f if edges_f.size > 2 else None

    qs_c = np.linspace(0.0, 1.0, N_QBINS_COARSE + 1)
    edges_c = np.quantile(d, qs_c)
    edges_c = np.unique(edges_c)
    edges_by_type_dir_coarse[t] = edges_c if edges_c.size > 2 else None


def _digitize_with_edges(dist_arr, edges):
    if edges is None:
        return None
    interior = edges[1:-1]
    if interior.size == 0:
        return None
    return np.digitize(dist_arr, interior, right=False).astype(np.int32)


def apply_qbins_per_type(df, edges_map, dist_col, out_col):
    out = np.full(len(df), SENTINEL_QBIN, dtype=np.int32)
    types = df["type"].astype("string").to_numpy()
    dist = df[dist_col].to_numpy(dtype="float64", copy=False)
    finite = np.isfinite(dist)

    for t in pd.unique(types):
        edges = edges_map.get(t, None)
        m = (types == t) & finite
        if not m.any():
            continue
        b = _digitize_with_edges(dist[m], edges)
        if b is None:
            continue
        out[m] = b

    df[out_col] = pd.Series(out, index=df.index, dtype="int32").astype("Int32")
    return df


# Type-based qbins for undirected distance
train_f = apply_qbins_per_type(train_f, edges_by_type, "dist", "dist_qbin_type")
test_f = apply_qbins_per_type(test_f, edges_by_type, "dist", "dist_qbin_type")
train_f = apply_qbins_per_type(
    train_f, edges_by_type_coarse, "dist", "dist_qbin_coarse_type"
)
test_f = apply_qbins_per_type(
    test_f, edges_by_type_coarse, "dist", "dist_qbin_coarse_type"
)

# Type-based qbins for directed distance (now correctly using directed edges)
train_f = apply_qbins_per_type(
    train_f, edges_by_type_dir, "dist_dir", "dist_qbin_dir_type"
)
test_f = apply_qbins_per_type(
    test_f, edges_by_type_dir, "dist_dir", "dist_qbin_dir_type"
)
train_f = apply_qbins_per_type(
    train_f, edges_by_type_dir_coarse, "dist_dir", "dist_qbin_dir_coarse_type"
)
test_f = apply_qbins_per_type(
    test_f, edges_by_type_dir_coarse, "dist_dir", "dist_qbin_dir_coarse_type"
)

N_QBINS_PAIR = 32
N_QBINS_PAIR_COARSE = 8
MIN_PAIR_SAMPLES = 80


def build_pair_edges(train_df, n_qbins, min_samples):
    edges_map = {}
    grp = train_df.groupby(["type", "atom_pair_0", "atom_pair_1"], observed=True)
    for key, g in grp:
        d = g["dist"].to_numpy(dtype="float64", copy=False)
        d = d[np.isfinite(d)]
        if d.size < min_samples:
            continue
        qs = np.linspace(0.0, 1.0, n_qbins + 1)
        edges = np.quantile(d, qs)
        edges = np.unique(edges)
        if edges.size > 2:
            edges_map[key] = edges
    return edges_map


pair_edges_fine = build_pair_edges(train_f, N_QBINS_PAIR, MIN_PAIR_SAMPLES)
pair_edges_coarse = build_pair_edges(train_f, N_QBINS_PAIR_COARSE, MIN_PAIR_SAMPLES)


def apply_qbins_per_pair_with_fallback(df, pair_edges_map, fallback_col, out_col):
    out = (
        df[fallback_col]
        .astype("Int32")
        .fillna(SENTINEL_QBIN)
        .to_numpy(dtype=np.int32, copy=True)
    )
    dist = df["dist"].to_numpy(dtype="float64", copy=False)
    finite = np.isfinite(dist)

    if len(pair_edges_map) == 0:
        df[out_col] = pd.Series(out, index=df.index, dtype="int32").astype("Int32")
        return df

    keys = list(pair_edges_map.keys())
    key_df = pd.DataFrame(keys, columns=["type", "atom_pair_0", "atom_pair_1"])
    key_df["type"] = key_df["type"].astype("string")
    key_df["atom_pair_0"] = key_df["atom_pair_0"].astype("string")
    key_df["atom_pair_1"] = key_df["atom_pair_1"].astype("string")

    tagged = (
        df[["type", "atom_pair_0", "atom_pair_1"]]
        .merge(
            key_df.assign(_has_pair_edges=1),
            on=["type", "atom_pair_0", "atom_pair_1"],
            how="left",
            sort=False,
        )["_has_pair_edges"]
        .to_numpy()
    )
    cand = (tagged == 1) & finite
    if not cand.any():
        df[out_col] = pd.Series(out, index=df.index, dtype="int32").astype("Int32")
        return df

    cand_df = df.loc[cand, ["type", "atom_pair_0", "atom_pair_1"]].copy()
    cand_df["_rowpos"] = np.flatnonzero(cand).astype(np.int64)

    for (t, ap0, ap1), g in cand_df.groupby(
        ["type", "atom_pair_0", "atom_pair_1"], observed=True, sort=False
    ):
        edges = pair_edges_map.get((t, ap0, ap1), None)
        if edges is None:
            continue
        rowpos = g["_rowpos"].to_numpy(dtype=np.int64, copy=False)
        b = _digitize_with_edges(dist[rowpos], edges)
        if b is None:
            continue
        out[rowpos] = b

    df[out_col] = pd.Series(out, index=df.index, dtype="int32").astype("Int32")
    return df


train_f = apply_qbins_per_pair_with_fallback(
    train_f, pair_edges_fine, "dist_qbin_type", "dist_qbin"
)
test_f = apply_qbins_per_pair_with_fallback(
    test_f, pair_edges_fine, "dist_qbin_type", "dist_qbin"
)
train_f = apply_qbins_per_pair_with_fallback(
    train_f, pair_edges_coarse, "dist_qbin_coarse_type", "dist_qbin_coarse"
)
test_f = apply_qbins_per_pair_with_fallback(
    test_f, pair_edges_coarse, "dist_qbin_coarse_type", "dist_qbin_coarse"
)

for col in [
    "type",
    "atom_pair_0",
    "atom_pair_1",
    "atom_0",
    "atom_1",
    "atom_0_canon",
    "atom_1_canon",
]:
    train_f[col] = train_f[col].astype("string")
    test_f[col] = test_f[col].astype("string")

print(
    train_f[
        [
            "type",
            "atom_0",
            "atom_1",
            "atom_0_canon",
            "atom_1_canon",
            "atom_pair_0",
            "atom_pair_1",
            "dist",
            "dist_dir",
            "dist_bin",
            "dist_qbin",
            "dist_qbin_coarse",
            "dist_qbin_dir_type",
            "dist_qbin_dir_coarse_type",
        ]
    ].head()
)

missing_train_atoms = (
    train_f["atom_pair_0"].isna().any()
    or train_f["atom_pair_1"].isna().any()
    or train_f["atom_0"].isna().any()
    or train_f["atom_1"].isna().any()
    or train_f["atom_0_canon"].isna().any()
    or train_f["atom_1_canon"].isna().any()
)
missing_test_atoms = (
    test_f["atom_pair_0"].isna().any()
    or test_f["atom_pair_1"].isna().any()
    or test_f["atom_0"].isna().any()
    or test_f["atom_1"].isna().any()
    or test_f["atom_0_canon"].isna().any()
    or test_f["atom_1_canon"].isna().any()
)
print("Any missing atoms in train?", bool(missing_train_atoms))
print("Any missing atoms in test?", bool(missing_test_atoms))
print("Missing dist in train:", int(train_f["dist"].isna().sum()))
print("Missing dist in test:", int(test_f["dist"].isna().sum()))
print("Missing dist_dir in train:", int(train_f["dist_dir"].isna().sum()))
print("Missing dist_dir in test:", int(test_f["dist_dir"].isna().sum()))



   type atom_0 atom_1 atom_0_canon atom_1_canon atom_pair_0 atom_pair_1  \
0  1JHC      H      C            C            H           C           H   
1  3JHC      H      C            C            H           C           H   
2  2JHH      H      H            H            H           H           H   
3  2JHH      H      H            H            H           H           H   
4  1JHC      H      C            C            H           C           H   

       dist  dist_dir  dist_bin  dist_qbin  dist_qbin_coarse  \
0  1.097524  1.097524        54         26                 6   
1  2.677621  2.677621       126          2                 0   
2  1.781957  1.781957        89         24                 6   
3  1.788658  1.788658        89         27                 6   
4  1.091999  1.091999        54         10                 2   

   dist_qbin_dir_type  dist_qbin_dir_coarse_type  
0                  53                         13  
1                   5                          1  
2          

In [4]:
global_median = float(train_f["scalar_coupling_constant"].median())

train_f["dist_qbin"] = train_f["dist_qbin"].astype("Int32").fillna(SENTINEL_QBIN)
train_f["dist_qbin_coarse"] = (
    train_f["dist_qbin_coarse"].astype("Int32").fillna(SENTINEL_QBIN)
)

train_f["dist_qbin_dir_type"] = (
    train_f["dist_qbin_dir_type"].astype("Int32").fillna(SENTINEL_QBIN)
)
train_f["dist_qbin_dir_coarse_type"] = (
    train_f["dist_qbin_dir_coarse_type"].astype("Int32").fillna(SENTINEL_QBIN)
)

g0d = train_f.groupby(
    ["type", "atom_0", "atom_1", "dist_qbin_dir_type"], observed=True
)["scalar_coupling_constant"].median()
g0dc = train_f.groupby(
    ["type", "atom_0", "atom_1", "dist_qbin_dir_coarse_type"], observed=True
)["scalar_coupling_constant"].median()

g0 = train_f.groupby(["type", "atom_0", "atom_1", "dist_qbin"], observed=True)[
    "scalar_coupling_constant"
].median()
g0c = train_f.groupby(["type", "atom_0", "atom_1", "dist_qbin_coarse"], observed=True)[
    "scalar_coupling_constant"
].median()

g0k = train_f.groupby(
    ["type", "atom_0_canon", "atom_1_canon", "dist_qbin"], observed=True
)["scalar_coupling_constant"].median()
g0kc = train_f.groupby(
    ["type", "atom_0_canon", "atom_1_canon", "dist_qbin_coarse"], observed=True
)["scalar_coupling_constant"].median()

g0k2 = train_f.groupby(["type", "atom_0_canon", "atom_1_canon"], observed=True)[
    "scalar_coupling_constant"
].median()
g0_2 = train_f.groupby(["type", "atom_0", "atom_1"], observed=True)[
    "scalar_coupling_constant"
].median()

g1 = train_f.groupby(
    ["type", "atom_pair_0", "atom_pair_1", "dist_qbin"], observed=True
)["scalar_coupling_constant"].median()

g1c = train_f.groupby(
    ["type", "atom_pair_0", "atom_pair_1", "dist_qbin_coarse"], observed=True
)["scalar_coupling_constant"].median()

g2 = train_f.groupby(["type", "atom_pair_0", "atom_pair_1"], observed=True)[
    "scalar_coupling_constant"
].median()

g2d = train_f.groupby(["type", "dist_qbin"], observed=True)[
    "scalar_coupling_constant"
].median()
g2dc = train_f.groupby(["type", "dist_qbin_coarse"], observed=True)[
    "scalar_coupling_constant"
].median()

g3 = train_f.groupby(["type"], observed=True)["scalar_coupling_constant"].median()


def hierarchical_predict(df):
    df = df.copy()

    df["type"] = df["type"].astype("string")
    for c in [
        "atom_0",
        "atom_1",
        "atom_0_canon",
        "atom_1_canon",
        "atom_pair_0",
        "atom_pair_1",
    ]:
        df[c] = df[c].astype("string")

    df["dist_qbin"] = df["dist_qbin"].astype("Int32").fillna(SENTINEL_QBIN)
    df["dist_qbin_coarse"] = (
        df["dist_qbin_coarse"].astype("Int32").fillna(SENTINEL_QBIN)
    )

    df["dist_qbin_dir_type"] = (
        df["dist_qbin_dir_type"].astype("Int32").fillna(SENTINEL_QBIN)
    )
    df["dist_qbin_dir_coarse_type"] = (
        df["dist_qbin_dir_coarse_type"].astype("Int32").fillna(SENTINEL_QBIN)
    )

    dq = df["dist_qbin"]
    dqc = df["dist_qbin_coarse"]
    dqd = df["dist_qbin_dir_type"]
    dqdc = df["dist_qbin_dir_coarse_type"]

    t_arr = df["type"].to_numpy()
    a0_arr = df["atom_0"].to_numpy()
    a1_arr = df["atom_1"].to_numpy()
    a0c_arr = df["atom_0_canon"].to_numpy()
    a1c_arr = df["atom_1_canon"].to_numpy()
    ap0_arr = df["atom_pair_0"].to_numpy()
    ap1_arr = df["atom_pair_1"].to_numpy()

    idx0d = pd.MultiIndex.from_arrays(
        [t_arr, a0_arr, a1_arr, dqd],
        names=["type", "atom_0", "atom_1", "dist_qbin_dir_type"],
    )
    idx0dc = pd.MultiIndex.from_arrays(
        [t_arr, a0_arr, a1_arr, dqdc],
        names=["type", "atom_0", "atom_1", "dist_qbin_dir_coarse_type"],
    )

    idx0 = pd.MultiIndex.from_arrays(
        [t_arr, a0_arr, a1_arr, dq],
        names=["type", "atom_0", "atom_1", "dist_qbin"],
    )
    idx0c = pd.MultiIndex.from_arrays(
        [t_arr, a0_arr, a1_arr, dqc],
        names=["type", "atom_0", "atom_1", "dist_qbin_coarse"],
    )

    idx0k = pd.MultiIndex.from_arrays(
        [t_arr, a0c_arr, a1c_arr, dq],
        names=["type", "atom_0_canon", "atom_1_canon", "dist_qbin"],
    )
    idx0kc = pd.MultiIndex.from_arrays(
        [t_arr, a0c_arr, a1c_arr, dqc],
        names=["type", "atom_0_canon", "atom_1_canon", "dist_qbin_coarse"],
    )
    idx0k2 = pd.MultiIndex.from_arrays(
        [t_arr, a0c_arr, a1c_arr],
        names=["type", "atom_0_canon", "atom_1_canon"],
    )
    idx0_2 = pd.MultiIndex.from_arrays(
        [t_arr, a0_arr, a1_arr],
        names=["type", "atom_0", "atom_1"],
    )

    idx1 = pd.MultiIndex.from_arrays(
        [t_arr, ap0_arr, ap1_arr, dq],
        names=["type", "atom_pair_0", "atom_pair_1", "dist_qbin"],
    )
    idx1c = pd.MultiIndex.from_arrays(
        [t_arr, ap0_arr, ap1_arr, dqc],
        names=["type", "atom_pair_0", "atom_pair_1", "dist_qbin_coarse"],
    )
    idx2 = pd.MultiIndex.from_arrays(
        [t_arr, ap0_arr, ap1_arr],
        names=["type", "atom_pair_0", "atom_pair_1"],
    )

    idx2d = pd.MultiIndex.from_arrays(
        [t_arr, dq],
        names=["type", "dist_qbin"],
    )
    idx2dc = pd.MultiIndex.from_arrays(
        [t_arr, dqc],
        names=["type", "dist_qbin_coarse"],
    )

    p = pd.Series(np.nan, index=df.index, dtype="float64")

    p0d = pd.Series(g0d.reindex(idx0d).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0d)

    p0dc = pd.Series(g0dc.reindex(idx0dc).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0dc)

    p0 = pd.Series(g0.reindex(idx0).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0)

    p0c = pd.Series(g0c.reindex(idx0c).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0c)

    p0k = pd.Series(g0k.reindex(idx0k).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0k)

    p0kc = pd.Series(g0kc.reindex(idx0kc).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0kc)

    p0k2 = pd.Series(g0k2.reindex(idx0k2).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0k2)

    p0_2 = pd.Series(g0_2.reindex(idx0_2).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p0_2)

    p1 = pd.Series(g1.reindex(idx1).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p1)

    p1c = pd.Series(g1c.reindex(idx1c).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p1c)

    p2 = pd.Series(g2.reindex(idx2).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p2)

    p2d = pd.Series(g2d.reindex(idx2d).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p2d)

    p2dc = pd.Series(g2dc.reindex(idx2dc).to_numpy(), index=df.index, dtype="float64")
    p = p.fillna(p2dc)

    p3 = pd.Series(df["type"].map(g3), index=df.index, dtype="float64")
    p = p.fillna(p3)

    p = p.fillna(global_median)
    return p


pred = hierarchical_predict(test_f)
assert pred.notna().all()

submission = pd.DataFrame(
    {"id": test["id"].values, "scalar_coupling_constant": pred.values}
)
submission["id"] = submission["id"].astype(sample_sub["id"].dtype, copy=False)

print(submission.head())
print(submission["scalar_coupling_constant"].describe())



        id  scalar_coupling_constant
0  2324604                 88.202350
1  2324605                 -0.953401
2  2324606                  2.878450
3  2324607                  2.878450
4  2324608                -11.328900
count    467813.000000
mean         14.191860
std          31.922231
min         -11.328900
25%          -0.953401
50%           2.878450
75%           3.687980
max          88.202350
Name: scalar_coupling_constant, dtype: float64


In [5]:
OUT_PATH = "submission.csv"
submission.to_csv(OUT_PATH, index=False, float_format="%.6f")
print("Wrote:", OUT_PATH)
print("File size (bytes):", os.path.getsize(OUT_PATH))

check = pd.read_csv(OUT_PATH)
assert list(check.columns) == ["id", "scalar_coupling_constant"]
assert len(check) == len(test)
assert check["scalar_coupling_constant"].notna().all()
print("Submission verified:", check.shape)

Wrote: submission.csv
File size (bytes): 8110042
Submission verified: (467813, 2)
